# Validación de integración — Consumo vs Resistencia (2000–2024)

En este notebook validamos la coherencia estructural entre:

- Consumo de antibióticos (ATC)
- Resistencia antimicrobiana (AMR)

Objetivo:

- Verificar que ambas fuentes pueden integrarse correctamente
- Comprobar coincidencias por:
  - year
  - country
  - atc_group
- Detectar posibles combinaciones faltantes antes de construir el análisis en Tableau

In [1]:
import pandas as pd
from pathlib import Path

# ==============================
# Rutas processed
# ==============================
PROCESSED_DIR = Path("../data/processed")

consumption_path = PROCESSED_DIR / "antibiotic_consumption_2012_2024_processed.csv"
resistance_path = PROCESSED_DIR / "antimicrobial_resistance_2012_2024_processed.csv"

# ==============================
# Cargar datasets
# ==============================
df_consumo = pd.read_csv(consumption_path)
df_resistencia = pd.read_csv(resistance_path)

print("📊 Consumo shape:", df_consumo.shape)
print("📊 Resistencia shape:", df_resistencia.shape)

display(df_consumo.head())
display(df_resistencia.head())

📊 Consumo shape: (1038, 4)
📊 Resistencia shape: (4186, 7)


,year,country,antibiotic_group,ddd_per_1000_inhabitants_per_day
0,2012,Belgium,J01D,1.9106
1,2012,Bulgaria,J01D,3.2477
2,2012,Croatia,J01D,4.0686
3,2012,Cyprus,J01D,5.3563
4,2012,Denmark,J01D,0.3228


,year,country_code,country,bacteria,antibiotic_group,resistance_percentage,atc_group
0,2012,BG,Bulgaria,Acinetobacter spp.,Aminoglycosides,58.461538,J01G
1,2012,CY,Cyprus,Acinetobacter spp.,Aminoglycosides,52.173913,J01G
2,2012,DE,Germany,Acinetobacter spp.,Aminoglycosides,5.882353,J01G
3,2012,DK,Denmark,Acinetobacter spp.,Aminoglycosides,10.389610,J01G
4,2012,EL,Greece,Acinetobacter spp.,Aminoglycosides,78.119935,J01G


## 2) Análisis de cardinalidad por clave estructural

Validamos el número de combinaciones únicas en cada dataset:

Consumo:
- year
- country
- antibiotic_group

Resistencia:
- year
- country
- atc_group
- bacteria

Esto nos permitirá entender el tipo de relación antes del merge.

In [2]:
# ==========================================================
# 1) Combinaciones únicas en consumo
# ==========================================================
consumo_unique = df_consumo[
    ["year", "country", "antibiotic_group"]
].drop_duplicates()

print("Combinaciones únicas en consumo:", consumo_unique.shape[0])

# ==========================================================
# 2) Combinaciones únicas en resistencia (con bacteria)
# ==========================================================
resistencia_unique_full = df_resistencia[
    ["year", "country", "atc_group", "bacteria"]
].drop_duplicates()

print("Combinaciones únicas en resistencia (con bacteria):", resistencia_unique_full.shape[0])

# ==========================================================
# 3) Combinaciones únicas en resistencia (sin bacteria)
# ==========================================================
resistencia_unique_no_bacteria = df_resistencia[
    ["year", "country", "atc_group"]
].drop_duplicates()

print("Combinaciones únicas en resistencia (sin bacteria):", resistencia_unique_no_bacteria.shape[0])

Combinaciones únicas en consumo: 1038
Combinaciones únicas en resistencia (con bacteria): 3420
Combinaciones únicas en resistencia (sin bacteria): 1161


## 3) Validación de solapamiento entre datasets

Comparamos las combinaciones únicas de:

- year
- country
- atc_group

Para identificar:

- Coincidencias en ambos datasets
- Combinaciones exclusivas de consumo
- Combinaciones exclusivas de resistencia

In [3]:
# ==========================================================
# Preparar claves para comparación
# ==========================================================

consumo_keys = consumo_unique.rename(columns={
    "antibiotic_group": "atc_group"
})

resistencia_keys = resistencia_unique_no_bacteria.copy()

# ==========================================================
# Merge tipo outer para ver coincidencias
# ==========================================================

comparison = consumo_keys.merge(
    resistencia_keys,
    on=["year", "country", "atc_group"],
    how="outer",
    indicator=True
)

print("Resumen de comparación:")
print(comparison["_merge"].value_counts())

display(comparison.head())

Resumen de comparación:
_merge
both          1035
right_only     126
left_only        3
Name: count, dtype: int64


,year,country,atc_group,_merge
0,2012,Austria,J01D,right_only
1,2012,Austria,J01G,right_only
2,2012,Austria,J01M,right_only
3,2012,Belgium,J01D,both
4,2012,Belgium,J01G,both


## 4) Identificación de países comunes

Para garantizar coherencia estructural,
nos quedamos únicamente con los países presentes en ambos datasets.

Esto evita:

- Filas con consumo sin resistencia
- Filas con resistencia sin consumo
- Problemas de interpretación en Tableau

Trabajaremos únicamente con la intersección de países.

In [4]:
# ==========================================================
# 1) Obtener países únicos en cada dataset
# ==========================================================
countries_consumo = set(df_consumo["country"].unique())
countries_resistencia = set(df_resistencia["country"].unique())

# ==========================================================
# 2) Intersección
# ==========================================================
common_countries = countries_consumo.intersection(countries_resistencia)

print("Número países en consumo:", len(countries_consumo))
print("Número países en resistencia:", len(countries_resistencia))
print("Número países comunes:", len(common_countries))

print("\nPaíses solo en consumo:")
print(sorted(countries_consumo - countries_resistencia))

print("\nPaíses solo en resistencia:")
print(sorted(countries_resistencia - countries_consumo))

Número países en consumo: 30
Número países en resistencia: 31
Número países comunes: 30

Países solo en consumo:
[]

Países solo en resistencia:
['Liechtenstein']


## 5) Filtrado a países comunes

Se filtran ambos datasets para conservar únicamente
los 30 países presentes en ambas fuentes.

Esto garantiza coherencia estructural antes del merge final.

In [5]:
# ==========================================================
# Filtrar datasets
# ==========================================================
df_consumo_common = df_consumo[
    df_consumo["country"].isin(common_countries)
].copy()

df_resistencia_common = df_resistencia[
    df_resistencia["country"].isin(common_countries)
].copy()

print("Shape consumo filtrado:", df_consumo_common.shape)
print("Shape resistencia filtrado:", df_resistencia_common.shape)

Shape consumo filtrado: (1038, 4)
Shape resistencia filtrado: (4166, 7)


## 6) Integración final (INNER JOIN)

Se realiza un merge tipo INNER por:

- year
- country
- atc_group

Esto garantiza que solo analizamos combinaciones donde
existen datos tanto de consumo como de resistencia.

In [6]:
# ==========================================================
# 1) Renombrar columna en consumo para coincidir con resistencia
# ==========================================================
df_consumo_common_renamed = df_consumo_common.rename(columns={
    "antibiotic_group": "atc_group"
})

# ==========================================================
# 2) Merge INNER
# ==========================================================
df_integrado = df_consumo_common_renamed.merge(
    df_resistencia_common,
    on=["year", "country", "atc_group"],
    how="inner"
)

print("Shape dataset integrado:", df_integrado.shape)

display(df_integrado.head())

Shape dataset integrado: (3748, 8)


,year,country,atc_group,ddd_per_1000_inhabitants_per_day,country_code,bacteria,antibiotic_group,resistance_percentage
0,2012,Belgium,J01D,1.9106,BE,Escherichia coli,Carbapenems,0.024278
1,2012,Belgium,J01D,1.9106,BE,Escherichia coli,Third-generation cephalosporins,6.883085
2,2012,Belgium,J01D,1.9106,BE,Klebsiella pneumoniae,Carbapenems,0.733945
3,2012,Belgium,J01D,1.9106,BE,Klebsiella pneumoniae,Third-generation cephalosporins,16.481481
4,2012,Bulgaria,J01D,3.2477,BG,Acinetobacter spp.,Carbapenems,60.344828


## 7) Validación estructural final

Verificamos que:

Para cada combinación:

- year
- country
- atc_group

El valor de consumo (ddd_per_1000_inhabitants_per_day)
es único y consistente.

In [8]:
# Comprobamos que consumo no cambia dentro de cada clave

check_consumo = (
    df_integrado
    .groupby(["year", "country", "atc_group"])["ddd_per_1000_inhabitants_per_day"]
    .nunique()
)

print("Máximo número de valores distintos de consumo por clave:")
print(check_consumo.max())

Máximo número de valores distintos de consumo por clave:
1


# ✅ Conclusión — Validación de integración

Tras el proceso de validación estructural:

## 1️⃣ Países
- Se identificó un país presente solo en resistencia (Liechtenstein).
- Se filtraron ambos datasets para trabajar únicamente con los 30 países comunes.

## 2️⃣ Cardinalidad
- Se validaron combinaciones únicas en consumo y resistencia.
- Se confirmó relación 1 → N (consumo → bacterias).

## 3️⃣ Integración
- Se realizó un INNER JOIN por:
  - year
  - country
  - atc_group

Resultado final:
- 3748 filas
- Modelo estructuralmente consistente
- Sin duplicación de valores de consumo

## 4️⃣ Validación final
Se comprobó que el valor de consumo es único por cada combinación:
(year, country, atc_group)

✔ Máximo número de valores distintos por clave = 1  
→ Merge correcto.

---

El dataset integrado está listo para análisis en Tableau.

## 8) Exportación del dataset integrado final

Se exporta el dataset final integrado que contiene:

- year
- country
- atc_group
- consumo (DDD)
- bacteria
- grupo antibiótico específico
- porcentaje de resistencia

Este archivo será la fuente única para Tableau.

In [9]:
# ==========================================================
# 1) Reordenar columnas (estructura lógica)
# ==========================================================
df_integrado_final = df_integrado[
    [
        "year",
        "country",
        "atc_group",
        "ddd_per_1000_inhabitants_per_day",
        "bacteria",
        "antibiotic_group",
        "resistance_percentage"
    ]
].copy()

print("Shape final dataset integrado:", df_integrado_final.shape)
display(df_integrado_final.head())


# ==========================================================
# 2) Exportar a processed
# ==========================================================
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "consumption_resistance_integrated_2012_2024.csv"
df_integrado_final.to_csv(output_path, index=False)

print(f"\n✅ Archivo exportado en: {output_path}")
print("Filas exportadas:", df_integrado_final.shape[0])
print("Columnas exportadas:", df_integrado_final.shape[1])

Shape final dataset integrado: (3748, 7)


,year,country,atc_group,ddd_per_1000_inhabitants_per_day,bacteria,antibiotic_group,resistance_percentage
0,2012,Belgium,J01D,1.9106,Escherichia coli,Carbapenems,0.024278
1,2012,Belgium,J01D,1.9106,Escherichia coli,Third-generation cephalosporins,6.883085
2,2012,Belgium,J01D,1.9106,Klebsiella pneumoniae,Carbapenems,0.733945
3,2012,Belgium,J01D,1.9106,Klebsiella pneumoniae,Third-generation cephalosporins,16.481481
4,2012,Bulgaria,J01D,3.2477,Acinetobacter spp.,Carbapenems,60.344828



✅ Archivo exportado en: ..\data\processed\consumption_resistance_integrated_2012_2024.csv
Filas exportadas: 3748
Columnas exportadas: 7
